## ✅ Conversion Validation Report — PASS

- **Workflow:** 03_formula_union_sort
- **Tools converted:** 7 (Low complexity, 0 joins, 2 formulas)
- **Self-correction iterations used:** 1 of 3

| Iteration | Mode | Result | Failures | Warnings | Action taken |
|---|---|---|---|---|---|
| 1 | standard | PASS | 0 | 0 | validated clean — exported |

# 03_formula_union_sort
Auto-generated by the Alteryx → Databricks Converter Engine
Source: 03_formula_union_sort.yxmd | Mode: standard
Altryx Accelerator 1.1.0
Complexity: Low (7 tools, 0 joins, 2 formulas)

In [ ]:
# SOURCE_TABLES: map tool ID, annotation, or path substring to a Unity Catalog table.
SOURCE_TABLES = {}
SOURCE_LABELS = {}
WORKFLOW_CONSTANTS = {}
def _resolve_source(tool_id):
    if SOURCE_TABLES.get(tool_id):
        return SOURCE_TABLES[tool_id]
    for key, value in SOURCE_TABLES.items():
        if value and (str(key) == tool_id or any(str(key).lower() in label.lower() for label in SOURCE_LABELS[tool_id])):
            return value
    raise ValueError('Map input ' + tool_id + ' in the SOURCE_TABLES configuration cell before running.')

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Input Sources

In [ ]:
# Tool 1: TextInput
_data_1 = [
    ('Widget', '1000'),
    ('Gadget', '2500'),
]
df_q1_sales_east = spark.createDataFrame(_data_1, ['Product', 'Q1Sales'])
df_q1_sales_east = df_q1_sales_east.withColumn("Q1Sales", F.col("Q1Sales").cast("long"))
# Tool 2: TextInput
_data_2 = [
    ('Widget', '800'),
    ('Sprocket', '1900'),
]
df_q1_sales_west = spark.createDataFrame(_data_2, ['Product', 'Q1Sales'])
df_q1_sales_west = df_q1_sales_west.withColumn("Q1Sales", F.col("Q1Sales").cast("long"))

## Transformations

In [ ]:
# Tool 3: Formula
df_tag_east_region = df_q1_sales_east
df_tag_east_region = df_tag_east_region.withColumn("Region", F.lit("East"))
df_tag_east_region = df_tag_east_region.withColumn("Region", F.col("Region").cast("string"))
df_tag_east_region = df_tag_east_region.withColumn("Tier", F.when(F.col("Q1Sales") > F.lit(1500), F.lit("High")).otherwise(F.lit("Standard")))
df_tag_east_region = df_tag_east_region.withColumn("Tier", F.col("Tier").cast("string"))
# Tool 4: Formula
df_tag_west_region = df_q1_sales_west
df_tag_west_region = df_tag_west_region.withColumn("Region", F.lit("West"))
df_tag_west_region = df_tag_west_region.withColumn("Region", F.col("Region").cast("string"))
df_tag_west_region = df_tag_west_region.withColumn("Tier", F.when(F.col("Q1Sales") > F.lit(1500), F.lit("High")).otherwise(F.lit("Standard")))
df_tag_west_region = df_tag_west_region.withColumn("Tier", F.col("Tier").cast("string"))
# Tool 5: Union
_union_streams_5 = [df_tag_east_region, df_tag_west_region]
from functools import reduce
df_combine_regions = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), _union_streams_5)
df_sort_by_sales_desc = df_combine_regions.orderBy(F.col("Q1Sales").desc(), F.col("Product").asc())  # Tool 6: Sort

## Output

In [ ]:
# Tool 7: OutputData -> main.gold.q1_sales_ranked
df_7 = df_sort_by_sales_desc
# df_7.write.format("delta").mode("overwrite").saveAsTable("main.gold.q1_sales_ranked")
df_7.createOrReplaceTempView("q1_sales_ranked")

## Validation

In [ ]:
# Row count sanity check
# Uncomment and set df_final to your final output DataFrame
# row_count = df_final.count()
# print(f"Final row count: {row_count:,}")
# assert row_count > 0, "Output DataFrame is empty!"
# df_final.limit(5).display()